Cell 1: setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "workspace"
schema = "de_ecommerce"
table = f"{catalog}.{schema}.gold_dim_customer"

add the SCD columns

Adds start_date, end_date, and is_current to the dimension, and marks every existing row as the current version.

In [0]:
spark.sql(f"ALTER TABLE {table} ADD COLUMNS (start_date DATE, end_date DATE, is_current BOOLEAN)")
spark.sql(f"UPDATE {table} SET start_date = DATE'2016-01-01', end_date = DATE'9999-12-31', is_current = true")
display(spark.table(table).limit(5))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7854939585622371>, line 1
----> 1 spark.sql(f"ALTER TABLE {table} ADD COLUMNS (start_date DATE, end_date DATE, is_current BOOLEAN)")
      2 spark.sql(f"UPDATE {table} SET start_date = DATE'2016-01-01', end_date = DATE'9999-12-31', is_current = true")
      3 display(spark.table(table).limit(5))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:919, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    916         _views.append(SubqueryAlias(df._plan, name))
    918 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 919 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    920 if "sql_command_result" in properties:
    921     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/

simulate the incoming batch

Creates 7 incoming records: 3 customers who moved to a São Paulo address, 2 unchanged, and 2 brand-new.

In [0]:
dim = spark.table(table)
cols = ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
        "customer_city", "customer_state", "customer_lat", "customer_lng"]

new_addr = (dim.filter(F.col("customer_state") == "SP")
               .orderBy("customer_key").first())

moved = (dim.filter(F.col("customer_state") != "SP")
            .orderBy("customer_key").limit(3)
            .select("customer_id", "customer_unique_id")
            .withColumn("customer_zip_code_prefix", F.lit(new_addr["customer_zip_code_prefix"]))
            .withColumn("customer_city", F.lit(new_addr["customer_city"]))
            .withColumn("customer_state", F.lit(new_addr["customer_state"]))
            .withColumn("customer_lat", F.lit(new_addr["customer_lat"]))
            .withColumn("customer_lng", F.lit(new_addr["customer_lng"]))
            .select(*cols))

unchanged = (dim.filter(F.col("customer_state") == "SP")
                .orderBy("customer_key").limit(2).select(*cols))

brand_new = spark.createDataFrame(
    [("new_cust_001", "new_unique_001", "01310", "sao paulo", "SP", -23.56, -46.65),
     ("new_cust_002", "new_unique_002", "01310", "sao paulo", "SP", -23.56, -46.65)],
    cols)

incoming = moved.unionByName(unchanged).unionByName(brand_new)
moved_ids = [r["customer_id"] for r in moved.collect()]
print("moved customers:", moved_ids)
display(incoming)

moved customers: ['000161a058600d5901f007fab4c27140', '0001fd6190edaaf884bcaf3d49edf079', '0002414f95344307404f0ace7a26f1d5']


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng
000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,06273,osasco,SP,-23.499062689784747,-46.76793857798514
0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,06273,osasco,SP,-23.499062689784747,-46.76793857798514
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,06273,osasco,SP,-23.499062689784747,-46.76793857798514
00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,06273,osasco,SP,-23.499062689784747,-46.76793857798514
000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,04841,sao paulo,SP,-23.738708912922274,-46.68933298902944
new_cust_001,new_unique_001,01310,sao paulo,SP,-23.56,-46.65
new_cust_002,new_unique_002,01310,sao paulo,SP,-23.56,-46.65


build the staged table

Compares the incoming rows with the current rows on customer_id, keeps only the changed and new ones, assigns new surrogate keys (highest + 1, 2, 3...), and creates the merge_key column. Changed customers get two rows, and new customers get one.

In [0]:
current = spark.table(table).filter("is_current = true")

joined = incoming.alias("i").join(current.alias("c"),
                                  F.col("i.customer_id") == F.col("c.customer_id"), "left")

changes = (joined
    .filter(F.col("c.customer_id").isNull()
            | ~F.col("i.customer_city").eqNullSafe(F.col("c.customer_city"))
            | ~F.col("i.customer_state").eqNullSafe(F.col("c.customer_state"))
            | ~F.col("i.customer_zip_code_prefix").eqNullSafe(F.col("c.customer_zip_code_prefix")))
    .select("i.*", F.col("c.customer_id").isNotNull().alias("is_existing")))

max_key = spark.table(table).agg(F.max("customer_key")).first()[0]

new_versions = (changes
    .withColumn("customer_key", F.lit(max_key) + F.row_number().over(Window.orderBy("customer_id")))
    .withColumn("effective_date", F.current_date()))

expire_rows = new_versions.filter("is_existing").withColumn("merge_key", F.col("customer_id"))
insert_rows = new_versions.withColumn("merge_key", F.lit(None).cast("string"))

staged = expire_rows.unionByName(insert_rows)
staged.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.tmp_customer_changes")
display(spark.table(f"{catalog}.{schema}.tmp_customer_changes").orderBy("customer_id", "merge_key"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,is_existing,customer_key,effective_date,merge_key
000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,06273,osasco,SP,-23.499062689784747,-46.76793857798514,true,99442,2026-10-27,null
000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,06273,osasco,SP,-23.499062689784747,-46.76793857798514,true,99442,2026-10-27,000161a058600d5901f007fab4c27140
0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,06273,osasco,SP,-23.499062689784747,-46.76793857798514,true,99443,2026-10-27,null
0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,06273,osasco,SP,-23.499062689784747,-46.76793857798514,true,99443,2026-10-27,0001fd6190edaaf884bcaf3d49edf079
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,06273,osasco,SP,-23.499062689784747,-46.76793857798514,true,99444,2026-10-27,null
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,06273,osasco,SP,-23.499062689784747,-46.76793857798514,true,99444,2026-10-27,0002414f95344307404f0ace7a26f1d5
new_cust_001,new_unique_001,01310,sao paulo,SP,-23.56,-46.65,false,99445,2026-10-27,null
new_cust_002,new_unique_002,01310,sao paulo,SP,-23.56,-46.65,false,99446,2026-10-27,null


the MERGE

Matches the staged rows to the current rows on customer_id = merge_key. A match closes the old row, and no match inserts a new row.

In [0]:
spark.sql(f"""
MERGE INTO {table} AS t
USING {catalog}.{schema}.tmp_customer_changes AS s
ON t.customer_id = s.merge_key AND t.is_current = true
WHEN MATCHED THEN
  UPDATE SET is_current = false, end_date = s.effective_date
WHEN NOT MATCHED THEN
  INSERT (customer_key, customer_id, customer_unique_id, customer_zip_code_prefix, customer_city,
          customer_state, customer_lat, customer_lng, start_date, end_date, is_current)
  VALUES (s.customer_key, s.customer_id, s.customer_unique_id, s.customer_zip_code_prefix, s.customer_city,
          s.customer_state, s.customer_lat, s.customer_lng, s.effective_date, DATE'9999-12-31', true)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

verify and clean up

Counts the rows, shows the history of the moved and new customers, then deletes the temporary table.

In [0]:
d = spark.table(table)
print("total rows:", d.count(), "| current rows:", d.filter("is_current = true").count())

display(d.filter(F.col("customer_id").isin(moved_ids + ["new_cust_001", "new_cust_002"]))
          .select("customer_key", "customer_id", "customer_city", "customer_state",
                  "start_date", "end_date", "is_current")
          .orderBy("customer_id", "start_date"))

spark.sql(f"DROP TABLE {catalog}.{schema}.tmp_customer_changes")

total rows: 99446 | current rows: 99443


customer_key,customer_id,customer_city,customer_state,start_date,end_date,is_current
2,000161a058600d5901f007fab4c27140,itapecerica,MG,2016-01-01,2026-10-27,false
99442,000161a058600d5901f007fab4c27140,osasco,SP,2026-10-27,9999-12-31,true
3,0001fd6190edaaf884bcaf3d49edf079,nova venecia,ES,2016-01-01,2026-10-27,false
99443,0001fd6190edaaf884bcaf3d49edf079,osasco,SP,2026-10-27,9999-12-31,true
4,0002414f95344307404f0ace7a26f1d5,mendonca,MG,2016-01-01,2026-10-27,false
99444,0002414f95344307404f0ace7a26f1d5,osasco,SP,2026-10-27,9999-12-31,true
99445,new_cust_001,sao paulo,SP,2026-10-27,9999-12-31,true
99446,new_cust_002,sao paulo,SP,2026-10-27,9999-12-31,true


DataFrame[]